# Band Structures of Bulk Solids

This notebook connects three complementary descriptions of bulk electronic structure: a self-consistent ground state, the density of states (DOS), and the dispersion of individual bands through reciprocal space. Silver (Ag) provides a simple face-centered-cubic metal, while iron (Fe) introduces a magnetic body-centered-cubic example.

### Learning goals

- calculate and save a periodic GPAW ground state;
- construct a DOS with energies referenced to the Fermi level;
- identify a crystal's Bravais lattice and special $k$-points;
- evaluate bands along a chosen high-symmetry path using a fixed density; and
- recognize when orbital, spin, convergence, and lattice-path information are needed for a defensible interpretation.

Energies are reported in electronvolts (eV), DOS values in states per eV for the computational cell, and reciprocal-space coordinates in the basis of the reciprocal lattice vectors. Unless stated otherwise, $E-E_{\mathrm{F}}=0$ marks the Fermi level. The numerical settings below are suitable for learning the workflow, but they should be converged before quantitative use.

## Imports

Matplotlib provides the plots, GPAW performs the plane-wave DFT calculations, and ASE supplies the bulk structures, reciprocal-space paths, and DOS interface.

In [1]:
import matplotlib.pyplot as plt
from gpaw import GPAW, PW

from ase.build import bulk
from ase.dft.dos import DOS

## Build the Ag primitive cell

`bulk('Ag')` constructs ASE's default bulk Ag structure: a one-atom primitive cell of the face-centered-cubic (FCC) crystal. The cell geometry determines both the real-space periodicity and the reciprocal lattice used later for the band path.

In [2]:
atoms = bulk('Ag')

## Bulk Ag ground-state calculation

The calculator uses a plane-wave basis with a kinetic-energy cutoff of 350 eV and an $8 \times 8 \times 8$ Monkhorst-Pack $k$-point mesh. The `Ag: 11` setup requests an Ag PAW dataset with 11 valence electrons. GPAW's text output is written separately so that convergence and calculator details remain auditable.

The cutoff, $k$-point mesh, electronic occupations, and lattice constant all affect the final energies and bands. Here they define one internally consistent learning calculation; they are not yet a convergence study.

In [3]:
calc = GPAW(
    mode=PW(350), kpts=[8, 8, 8], txt='../files/advanced/tutorial_06/gpaw.bulk_Ag.txt', setups={'Ag': '11'}
)

### Attach the calculator and trigger self-consistency

ASE calculations are lazy: assigning `atoms.calc` only connects the objects, while requesting the potential energy starts the self-consistent-field calculation. The printed value is the total potential energy of this one-atom periodic cell. It is not a cohesive energy, and absolute totals from different elements should not be compared directly.

In [ ]:
atoms.calc = calc
print(
    'Bulk {0} potential energy = {1:.3f}eV'.format(
        atoms.get_chemical_formula(), atoms.get_potential_energy()
    )
)

### Save the converged state

The `.gpw` restart file stores the converged calculator state needed by the later DOS and fixed-density calculations. Reusing it avoids repeating the ground-state self-consistency cycle and ensures that the analyses share the same electronic density.

In [ ]:
ground_state_file = '../files/advanced/tutorial_06/bulk_Ag_groundstate.gpw'
calc.write(ground_state_file)

## Density of states

The DOS counts how many one-electron states occur in each energy interval:

$$
g(E)=\sum_{n,\mathbf{k}} w_{\mathbf{k}}\,\delta(E-\varepsilon_{n\mathbf{k}}).
$$

ASE shifts the eigenvalues by the Fermi energy, so $E-E_{\mathrm{F}}=0$ separates occupied and unoccupied states at zero temperature. With `width=0`, ASE uses linear tetrahedron interpolation on the regular $k$-point grid instead of adding Gaussian broadening. The 800 points control how finely the resulting curve is sampled; they do not replace convergence with respect to the underlying $k$-point mesh.

In [ ]:
calc = GPAW(ground_state_file)
dos = DOS(calc, npts=800, width=0)
energies = dos.get_energies()
weights = dos.get_dos()

### Plot the Ag DOS

The vertical dashed line marks $E-E_{\mathrm{F}}=0$. States to its left are occupied in the ground-state picture; states to its right are unoccupied Kohn-Sham states.

In [ ]:
fig, ax = plt.subplots()
ax.axvline(0.0, linestyle='--', color='black', alpha=0.5)
ax.plot(energies, weights)
ax.set_xlabel('Energy - Fermi Energy (eV)')
ax.set_ylabel('Density of States (1/eV)')
fig.tight_layout()

### Interpreting the Ag DOS

A useful first question is which parts of the spectrum are mainly $s$-like and which are mainly $d$-like. Localized atomic $d$ orbitals generally overlap less strongly between neighboring atoms than the more delocalized $s$ orbitals. In a solid, weaker overlap produces a comparatively narrow $d$ band, while stronger overlap produces a broader $sp$-derived band. The fivefold orbital degeneracy of the $d$ manifold also helps make its DOS large.

For this Ag result, the prominent occupied feature from approximately -6.2 to -2.6 eV is expected to be predominantly $4d$-derived. The lower-amplitude states spread across a wider energy range, including the states crossing $E_{\mathrm{F}}$, are expected to contain more $5s$ and $5p$ character. This is a chemically motivated assignment, not a direct measurement: a projected DOS is required to separate the $s$, $p$, and $d$ contributions explicitly.

Ag is a noble metal, and its $d$ manifold lies below the Fermi level while a more dispersive band crosses $E_{\mathrm{F}}$. By contrast, many other transition metals have a partially occupied $d$ manifold and therefore substantial $d$ character at the Fermi level.

> **Caution about unoccupied states:** the states above $E_{\mathrm{F}}$ are Kohn-Sham eigenvalues evaluated for a ground-state density. Their number and high-energy accuracy depend on the requested bands and basis settings, and ordinary ground-state DFT eigenvalues are not quasiparticle excitation energies. Near-Fermi trends can still be informative, but high unoccupied states should not be interpreted quantitatively without additional convergence checks and, where needed, a more appropriate excited-state method.

## Reciprocal space and band structure

The DOS integrates information across the Brillouin zone, whereas a band structure retains the wave-vector dependence $\varepsilon_n(\mathbf{k})$. It therefore reveals which bands are flat or dispersive, where bands cross the Fermi level, and how crystal symmetry organizes the electronic states.

The next cell reconstructs the Ag cell and asks ASE to identify its Bravais lattice, special reciprocal-space points, and a conventional default path.

In [ ]:
atoms = bulk('Ag')

lat = atoms.cell.get_bravais_lattice()
print(lat.description())

### Inspect the first Brillouin zone

The Brillouin-zone drawing places the named special points on the FCC reciprocal-space polyhedron. These labels are lattice-specific: the same letter sequence must not be transferred to a structure with a different Bravais lattice.

In [ ]:
lat.plot_bz(show=True)
plt.show()

### Define a high-symmetry path

The path `WLGXWK` means $W \rightarrow L \rightarrow \Gamma \rightarrow X \rightarrow W \rightarrow K$ for the FCC cell. `density=10` controls the sampling density along this plotting path; it is separate from the uniform $8 \times 8 \times 8$ mesh used to converge the ground-state density. The path is saved as JSON so that its geometry can be inspected or reused.

In [ ]:
path = atoms.cell.bandpath('WLGXWK', density=10)
path.write('../files/advanced/tutorial_06/path.json')
print(path)

### Visualize the selected route

Plotting the path before the electronic calculation is a useful geometry check: the connected segments should pass through the intended FCC special points in the intended order.

In [ ]:
path.plot()
plt.show()

### Evaluate bands at fixed density

The converged Ag ground state is reloaded and diagonalized non-self-consistently at the $k$-points along the path. The electron density remains fixed, which is efficient because the path is intended for interpolation and visualization rather than for Brillouin-zone integration. Turning symmetry off preserves the explicitly requested sequence of path points.

This separation is important: a dense uniform mesh determines the ground-state density, while a connected high-symmetry path displays the resulting eigenvalue dispersion.

In [ ]:
ground_state_file = '../files/advanced/tutorial_06/bulk_Ag_groundstate.gpw'
calc = GPAW(ground_state_file)
calc = calc.fixed_density(kpts=path, symmetry='off')

### Collect and save the Ag bands

`band_structure()` packages the path, eigenvalues, and reference energy into an ASE `BandStructure` object. Saving it to JSON decouples plotting from the GPAW calculation and makes the numerical band data easy to revisit.

In [ ]:
bs = calc.band_structure()
bs.write('../files/advanced/tutorial_06/bs.json')
print(bs)

### Plot and inspect the Ag dispersion

The inline plot shows how the eigenvalues change along the FCC path. A band crossing the Fermi reference is the reciprocal-space signature of metallic behavior. Relatively flat groups of bands contribute many states within a narrow energy interval and therefore tend to coincide with large DOS features; strongly dispersive bands spread their states over a wider energy range.

The saved file can also be plotted from a terminal. From the notebook directory, use:

```bash
ase band-structure ../files/advanced/tutorial_06/bs.json
```

The shorter command `ase band-structure bs.json` works only when the current directory already contains `bs.json`. For a direct comparison with the DOS, label or transform the band-energy axis consistently so that the Fermi reference corresponds to $E-E_{\mathrm{F}}=0$.

In [ ]:
ax = bs.plot()
ax.set_ylim(-2.0, 30.0)
plt.show()

## BCC Fe: a lattice-specific reciprocal-space path

A band structure samples eigenvalues along a path through reciprocal space, and the conventional high-symmetry points depend on the real-space Bravais lattice. Ag is FCC, whereas `bulk('Fe')` constructs BCC Fe. The FCC Ag path therefore cannot serve as the standard path for Fe.

GPAW can mathematically evaluate Fe eigenvalues at coordinates copied from the Ag calculation, so the calculation may still finish. The problem is crystallographic: FCC labels such as $W$, $L$, and $X$ do not describe the standard BCC symmetry directions, making the plotted x-axis physically misleading.

The cells below ask ASE to identify the Fe lattice and generate its standard path directly from the Fe cell. For BCC, ASE uses the special points $\Gamma$, $H$, $P$, and $N$. The same `path_fe` object is then passed to the fixed-density band calculation, removing any dependency on the earlier Ag path.

In [ ]:
atoms_fe = bulk('Fe')

lattice_fe = atoms_fe.cell.get_bravais_lattice()
print(lattice_fe.description())

path_fe = atoms_fe.cell.bandpath(npoints=100)
path_fe.write('../files/advanced/tutorial_06/path_Fe.json')
print(f'Fe high-symmetry path: {path_fe.path}')
_, _, fe_special_point_labels = path_fe.get_linear_kpoint_axis()
print('Fe special-point labels:', ' -> '.join(fe_special_point_labels))
print(path_fe)

ax = path_fe.plot()
ax.set_title('ASE-generated high-symmetry path for BCC Fe')
plt.show()

### Spin-polarized Fe ground state

BCC Fe is ferromagnetic. A non-spin-polarized calculation would constrain the two collinear spin channels to the same eigenvalues and miss the exchange splitting. In a magnetic solution,

$$
E_{n\uparrow}(\mathbf{k}) \neq E_{n\downarrow}(\mathbf{k}).
$$

A nonzero initial moment gives the SCF cycle a magnetic starting density, while `spinpol=True` explicitly retains two spin channels. The converged total moment describes the net spin magnetization of the periodic cell. Because this primitive cell contains one Fe atom, the cell moment divided by the atom count is also the moment per Fe atom; it is not an atom-projected local moment.

In [ ]:
atoms_fe.set_initial_magnetic_moments([2.2] * len(atoms_fe))

calc_fe = GPAW(
    mode=PW(350),
    kpts=[8, 8, 8],
    spinpol=True,
    txt='../files/advanced/tutorial_06/gpaw.bulk_Fe.txt',
)
atoms_fe.calc = calc_fe

total_energy_fe = atoms_fe.get_potential_energy()
total_magnetic_moment_fe = atoms_fe.get_magnetic_moment()
magnetic_moment_per_fe = total_magnetic_moment_fe / len(atoms_fe)
fermi_level_fe = calc_fe.get_fermi_level()

print(f'Bulk Fe total energy = {total_energy_fe:.3f} eV')
print(f'Spin channels = {calc_fe.get_number_of_spins()}')
print(f'Total magnetic moment = {total_magnetic_moment_fe:.3f} Bohr magnetons per cell')
print(f'Magnetic moment = {magnetic_moment_per_fe:.3f} Bohr magnetons per Fe atom')
print(f'Fermi level = {fermi_level_fe:.3f} eV')

ground_state_file_fe = '../files/advanced/tutorial_06/bulk_Fe_groundstate.gpw'
calc_fe.write(ground_state_file_fe)

### Reading the magnetic ground-state output

The reported two spin channels and nonzero total moment confirm that the SCF cycle converged to a collinear magnetic solution. The moment is a calculated result, not a fixed input: its exact value depends on the lattice constant, exchange-correlation approximation, basis cutoff, $k$-point sampling, occupations, and convergence thresholds. The printed Fermi level uses GPAW's internal energy reference; the DOS and band plots below subtract it so that the physically useful reference is $E-E_{\mathrm{F}}=0$.

### Spin-resolved Fe density of states

ASE numbers the two collinear channels as spin indices 0 and 1; these are conventionally displayed as spin up and spin down relative to the chosen quantization axis. The index alone should not be treated as a universal majority/minority label without checking the magnetization and occupations.

The plot places spin index 0 above the axis and the negative of spin index 1 below it. The negative sign is only a visual convention: both physical DOS values are nonnegative. ASE has already shifted the DOS energy grid so that $E-E_{\mathrm{F}}=0$.

In [ ]:
calc_fe_dos = GPAW(ground_state_file_fe, txt=None)
dos_fe = DOS(calc_fe_dos, npts=800, width=0)
energies_fe = dos_fe.get_energies()
dos_spin_0_fe = dos_fe.get_dos(spin=0)
dos_spin_1_fe = dos_fe.get_dos(spin=1)

if calc_fe_dos.get_number_of_spins() != 2:
    raise RuntimeError('Expected two spin channels for ferromagnetic Fe.')

dos_window_fe = (energies_fe >= -8.0) & (energies_fe <= 6.0)
dos_limit_fe = 1.1 * max(
    dos_spin_0_fe[dos_window_fe].max(),
    dos_spin_1_fe[dos_window_fe].max(),
)

fig, ax = plt.subplots()
ax.axvline(0.0, linestyle='--', color='black', linewidth=1.0)
ax.axhline(0.0, color='0.5', linewidth=0.8)
ax.plot(energies_fe, dos_spin_0_fe, label='spin index 0 (up)')
ax.plot(energies_fe, -dos_spin_1_fe, label='spin index 1 (down)')
ax.set_xlim(-8.0, 6.0)
ax.set_ylim(-dos_limit_fe, dos_limit_fe)
ax.set_xlabel(r'Energy relative to $E_F$ (eV)')
ax.set_ylabel('DOS (states/eV per cell)')
ax.set_title('Spin-resolved DOS of BCC Fe')
ax.legend()
fig.tight_layout()
plt.show()

### Interpreting the Fe DOS

The two channels have different peak positions and weights, which is the DOS signature of exchange splitting. Both channels retain states near $E_{\mathrm{F}}$, consistent with metallic Fe. The mirrored presentation makes the difference easy to see, but it does not by itself establish a universal majority/minority label for a spin index.

### Spin-resolved Fe band structure

The fixed-density calculation diagonalizes the magnetic ground-state Hamiltonian along `path_fe` without repeating the density optimization. Both spin channels are retained. Subtracting the stored reference moves the Fermi level to 0 eV while preserving ASE's BCC high-symmetry labels on the x-axis.

In [ ]:
calc_fe_ground = GPAW(ground_state_file_fe, txt=None)
calc_fe_bands = calc_fe_ground.fixed_density(
    kpts=path_fe,
    symmetry='off',
    txt='../files/advanced/tutorial_06/gpaw.band_Fe.txt',
)

bs_fe = calc_fe_bands.band_structure().subtract_reference()
bs_fe.write('../files/advanced/tutorial_06/bs_Fe.json')
_, _, fe_band_labels = bs_fe.get_labels()

if bs_fe.energies.shape[0] != 2:
    raise RuntimeError('Expected two spin channels in the Fe band structure.')

print(f'Fe band path: {path_fe.path}')
print('Fe band labels:', ' -> '.join(fe_band_labels))
print(f'Band energy reference = {bs_fe.reference:.1f} eV')
print(f'Band array shape (spin, k-point, band) = {bs_fe.energies.shape}')

In [ ]:
ax = bs_fe.plot(
    emin=-15.0,
    emax=25.0,
    colors=['tab:blue', 'tab:orange'],
    label=['spin index 0 (up)', 'spin index 1 (down)'],
    ylabel=r'Energy relative to $E_F$ (eV)',
)
ax.set_title('Spin-resolved bands of BCC Fe')
ax.figure.tight_layout()
plt.show()

### Interpreting the Fe bands

The blue and orange bands differ because the magnetic ground state produces different effective potentials for the two spin channels. Bands from both channels cross $E_{\mathrm{F}}=0$, again identifying Fe as metallic. The x-axis follows the ASE-generated BCC sequence $\Gamma-H-N-\Gamma-P-H$ followed by the separate $P-N$ segment; the combined $H,P$ tick marks the break between those two path segments.

## Comparing Ag and Fe

- **Ag:** FCC and approximately nonmagnetic in this treatment. Its bands are spin-degenerate, and the occupied narrow DOS feature is associated mainly with the filled $d$ manifold.
- **Fe:** BCC and ferromagnetic. Its nonzero magnetic moment accompanies exchange-split spin channels, so separate DOS and band curves are required.

The two workflows connect real-space structure to electronic structure in the sequence `lattice -> Brillouin zone -> high-symmetry path -> bands`. The Fe workflow adds a second connection: `magnetic ground state -> spin-dependent DOS and bands`. Absolute Ag and Fe total energies should not be compared because they describe different elements and reference energies.

## Limitations

This remains a tutorial-scale calculation rather than a converged model of experimental Fe:

- the ground-state density uses a finite $8 \times 8 \times 8$ $k$-point mesh;
- the plane-wave basis is limited to a 350 eV cutoff;
- the result depends on GPAW's default LDA exchange-correlation treatment;
- no systematic cutoff, $k$-point, occupation, or lattice-constant convergence study is included; and
- the Fe moment, DOS peaks, and band positions can shift with the lattice constant and numerical settings.

The unoccupied Kohn-Sham bands are also not quasiparticle excitation energies. The figures should therefore be interpreted as a reproducible demonstration of lattice and spin effects, not as proof of quantitative agreement with measured Fe spectra.

## Takeaways and possible extensions

- Generate every band path from the cell whose electronic structure is being calculated. A mathematically valid set of $k$-points can still carry the wrong crystallographic labels.
- Explicit magnetic initialization and two spin channels expose the exchange splitting that a non-spin-polarized Fe calculation would miss.
- Referencing both DOS and bands to $E_{\mathrm{F}}=0$ makes their near-Fermi features directly comparable.
- The next useful additions are orbital-projected spin DOS, a small cutoff and $k$-point convergence check, comparison of relaxed and reference lattice constants, and spin-orbit coupling where relevant.